# Assamese ASR -> ONNX (for the local IndicVoice pipeline)

Run this notebook top to bottom on Google Colab (**Runtime > Change runtime type > T4 GPU** makes it faster, CPU also works).

At the end you get one file, `assamese-asr.zip`. Download it and place the **extracted folder** at:

```
Voice model/models/asr-as-ctc/
    model.onnx
    tokens.json
    meta.json
```

Nothing else is needed: the local pipeline picks it up automatically for Assamese and falls back to the Bengali model if the folder is missing.

Total time: ~5-10 minutes on the free tier.

In [ ]:
!pip -q install "transformers>=4.40" "torch>=2.2" onnx onnxruntime soundfile librosa
print('deps ready')

In [ ]:
import json, os, torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2FeatureExtractor

# tried in order; the first one that loads wins
CANDIDATES = [
    "Harveenchadha/vakyansh-wav2vec2-assamese-asm-8",
    "infinitejoy/wav2vec2-large-xls-r-300m-assamese-cv8",
    "StephennFernandes/wav2vec2-XLS-R-300m-assamese",
    "manandey/wav2vec2-large-xlsr-assamese",
]

model_id = None
for candidate in CANDIDATES:
    try:
        model = Wav2Vec2ForCTC.from_pretrained(candidate)
        fe = Wav2Vec2FeatureExtractor.from_pretrained(candidate)
        model_id = candidate
        break
    except Exception as exc:
        print(f"skip {candidate}: {str(exc)[:120]}")
if model_id is None:
    raise SystemExit("no Assamese model could be loaded")
print("using", model_id)

model.eval()
blank_id = model.config.pad_token_id if model.config.pad_token_id is not None else 0
vocab = model.config.vocab_size
print("vocab size:", vocab, "| blank:", blank_id, "| do_normalize:", getattr(fe, 'do_normalize', True))

In [ ]:
os.makedirs("export", exist_ok=True)
dummy = torch.randn(1, 32000 * 8)          # 8 s of 16 kHz audio

torch.onnx.export(
    model, dummy, "export/model_fp32.onnx",
    input_names=["input_values"], output_names=["logits"],
    dynamic_axes={"input_values": {0: "batch", 1: "samples"},
                  "logits": {0: "batch", 1: "frames"}},
    opset_version=17, do_constant_folding=True,
)
print("fp32 size MB:", round(os.path.getsize("export/model_fp32.onnx") / 1e6, 1))

from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic("export/model_fp32.onnx", "export/model.onnx", weight_type=QuantType.QInt8)
print("int8 size MB:", round(os.path.getsize("export/model.onnx") / 1e6, 1))

# id -> token list, so the local loader never has to guess the vocab layout
id2tok = model.config.id2label
tokens = [str(id2tok.get(i, "")) for i in range(vocab)]
tokens = ["<pad>" if t in ("<pad>", "") else t for t in tokens]
json.dump(tokens, open("export/tokens.json", "w"), ensure_ascii=False)
json.dump({"sample_rate": 16000, "normalize": bool(getattr(fe, "do_normalize", True)),
           "blank_id": int(blank_id), "word_delimiter": "|", "model_id": model_id},
          open("export/meta.json", "w"), indent=2)
print("tokens:", tokens[:8], "...", tokens[-4:])

In [ ]:
# sanity check: the quantised model must reproduce the fp32 transcript
import numpy as np, onnxruntime as ort

sess = ort.InferenceSession("export/model.onnx", providers=["CPUExecutionProvider"])
wave = np.random.RandomState(0).randn(16000 * 4).astype(np.float32) * 0.1
norm = (wave - wave.mean()) / np.sqrt(wave.var() + 1e-7) if getattr(fe, "do_normalize", True) else wave
logits = sess.run(None, {"input_values": norm[None, :]})[0]
ids = logits.argmax(-1)[0]

out, prev = [], -1
for i in ids:
    if i != prev and i != blank_id:
        out.append(tokens[i])
    prev = i
print("logits shape:", logits.shape)
print("decode sample:", "".join(out).replace("|", " ")[:80] or "(silence - expected on noise)")
assert logits.shape[0] == 1 and logits.shape[2] == vocab, "unexpected model output shape"

In [ ]:
import shutil
shutil.make_archive("assamese-asr", "zip", "export")
print("created assamese-asr.zip")
try:
    from google.colab import files
    files.download("assamese-asr.zip")
except Exception:
    print("not on Colab: the zip is in export/")